In [1]:
# I have removed, because of github. if you want to use it agian, you need to import
import numpy as np

from tqdm import tqdm
from transformers.pipelines.pt_utils import KeyDataset
from datasets import load_dataset
from transformers import pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer


c:\Users\trt\Desktop\research_pyt\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load our data
data = load_dataset("rotten_tomatoes")
data

In [ ]:
data["train"][0, -1]

# Using a Task-Specific Model

In [ ]:
# Path to our HF model
model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"

In [ ]:
# Load model into pipeline
pipe = pipeline(
    model=model_path,
    tokenizer=model_path,
    return_all_scores=True,
    device="cuda:0"
)

In [ ]:
# Run inference
y_pred = []
for output in tqdm(pipe(KeyDataset(data["test"], "text")),
total=len(data["test"])):
    negative_score = output[0]["score"]
    positive_score = output[2]["score"]
    assignment = np.argmax([negative_score, positive_score])
    y_pred.append(assignment)

In [ ]:
def evaluate_performance(y_true, y_pred):
    """Create and print the classification report"""
    performance = classification_report(y_true, y_pred,target_names=["Negative Review", "Positive Review"])
    
    print(performance)

In [ ]:
evaluate_performance(data["test"]["label"], y_pred)

# Supervised Classification

In [ ]:
# Load model
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

In [ ]:
# Convert text to embeddings
train_embeddings = model.encode(data["train"]["text"], show_progress_bar=True)
test_embeddings = model.encode(data["test"]["text"], show_progress_bar=True)

In [ ]:
train_embeddings.shape

In [ ]:
# Train a logistic regression on our train embeddings
clf = LogisticRegression(random_state=42)
result = clf.fit(train_embeddings, data["train"]["label"])
print("Training finished!")

In [ ]:
y_pred = clf.predict(test_embeddings)
evaluate_performance(data["test"]["label"], y_pred)

# What If We Do Not Have Labeled Data?

In [ ]:
# Create embeddings for our labels
label_embeddings = model.encode(["A negative review", "A positive review"])

In [ ]:
# Find the best matching label for each document
sim_matrix = cosine_similarity(test_embeddings, label_embeddings)
y_pred = np.argmax(sim_matrix, axis=1)

In [ ]:
evaluate_performance(data["test"]["label"], y_pred)

# Text Classification with Generative Models
Using the Text-to-Text Transfer Transformer

In [ ]:
# Load our model
pipe = pipeline(
    "text2text-generation",
    model="google/flan-t5-small",
    device="cuda:0"
)

In [ ]:
# Prepare our data
prompt = "Is the following sentence positive or negative? "
data = data.map(lambda example: {"t5": prompt + example['text']})
data

In [ ]:
# Run inference
y_pred = []

for output in tqdm(pipe(KeyDataset(data["test"], "t5")),
total=len(data["test"])):
    text = output[0]["generated_text"]
    y_pred.append(0 if text == "negative" else 1)

In [ ]:
evaluate_performance(data["test"]["label"], y_pred)

# ChatGPT for Classification

Using this client, we create the chatgpt_generation function, which allows us to generate some text based on a specific prompt, input document, and the selected model: